# 📘Projet 1: Machine Learning Supervisé (Régression)

# 1.Importing and exploration of Data 

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("D:/AW PRV/AUTO FORMATION/Projets/kc_house_data.csv")
df

In [ ]:
# Exemple
df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S')

# Vérifier
print(df['date'].head())

In [ ]:
df['transaction_year'] = df['date'].dt.year

In [ ]:
df['house_age'] = df['transaction_year'] - df['yr_built']

In [ ]:
# Poser la colonne 'date' comme index
df.set_index('date', inplace=True)

# Vérifier
print(df.head())
print(df.index)  # Affiche l'index datetime

In [ ]:
# Trier le DataFrame par index (date)
df = df.sort_index()

# Vérifier
print(df.head())    # début du DataFrame (dates les plus anciennes)
print(df.tail())    # fin du DataFrame (dates les plus récentes)

In [ ]:
# Nombre de dates uniques vs total
total_dates = len(df)
unique_dates = df.index.nunique()

print(f"Total de lignes : {total_dates}")
print(f"Nombre de dates uniques : {unique_dates}")
print(f"Nombre de répétitions : {total_dates - unique_dates}")

In [ ]:
type(df.index)

In [ ]:
# 1. Créer un ID virtuel pour distinguer les observations d'une même date
df["virtual_id"] = df.groupby(level=0).cumcount()

# 2. Créer un MultiIndex (Date + ID virtuel)
df = df.set_index("virtual_id", append=True).sort_index()

# 3. Créer un calendrier journalier complet
daily_calendar = pd.date_range(
    start=df.index.get_level_values(0).min(),
    end=df.index.get_level_values(0).max(),
    freq="D"
)

# 4. Reindexer uniquement sur la dimension temporelle (Date)
df = df.reindex(daily_calendar, level=0)

In [ ]:
df.index.names
df.index.is_unique

In [ ]:
import matplotlib.pyplot as plt

presence = df.notna().any(axis=1).astype(int)

plt.figure(figsize=(12, 2))
plt.scatter(
    presence.index.get_level_values(0),
    presence,
    s=5
)

plt.yticks([0, 1], ["Absence", "Présence"])
plt.title("Présence des observations dans le temps")
plt.xlabel("Temps")
plt.tight_layout()
plt.show()

In [ ]:
df

# 2.Data Cleaning & Feature inginiering

In [ ]:
df = df.drop(columns=["id", "transaction_year"])

In [ ]:
print("Dimensions :", df.shape)
print("Nombre de features :", df.shape[1])
print("Nombre total de cellules :", df.size)
df.info()

In [ ]:
import numpy as np

df['renovated'] = np.where(
    df['yr_renovated'] == 0,
    df['yr_built'],      # si pas rénovée, on met l'année de construction
    df['yr_renovated']   # sinon on garde l'année de rénovation
)

In [ ]:
df

In [ ]:
df["total_rooms"] = df["bedrooms"] + df["bathrooms"]

In [ ]:
df['total_rooms'].fillna(0, inplace=True)
df.isnull().sum()
df["M_sqft"]=df["sqft_living"] / df["total_rooms"]
df.head()

In [ ]:
df["R_Prliv"]=df["price"] / df["sqft_living"]
df.head()

In [ ]:
df["basement_ratio"] = df["sqft_basement"] / df["sqft_living"]

In [ ]:
df["lot_ratio"] = df["sqft_lot"] / df["sqft_living"]

In [ ]:
df["lot_Sum"] = df["sqft_lot"] + df["sqft_living"]

In [ ]:
df["lot_Dif"] = df["sqft_lot"] - df["sqft_living"]

In [ ]:
df["isrenovat"] = (df["yr_renovated"] > 0).astype(int)

In [ ]:
df = df.drop(columns=["yr_renovated"])

In [ ]:
df

In [ ]:
#pip install folium

In [ ]:
import folium

# Centrer la carte sur Seattle
map_center = [df['lat'].mean(), df['long'].mean()]
m = folium.Map(location=map_center, zoom_start=11)

# Ajouter les maisons comme cercles
for idx, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=max(row['sqft_lot15']/5000, 1),  # taille selon lot
        color='blue',
        fill=True,
        fill_opacity=0.4,
        popup=f"sqft_living15: {row['sqft_living15']}"
    ).add_to(m)

m  # Affiche la carte interactive dans Jupyter

In [ ]:
# Liste des variables qualitatives
qual_vars = ['waterfront', 'view', 'condition', 'grade', 'zipcode', 'isrenovat']

# Afficher les catégories uniques pour chaque variable
for var in qual_vars:
    print(f"Variable : {var}")
    print("Catégories uniques :", df[var].unique())
    print("Valeur et fréquence :\n", df[var].value_counts())
    print("-"*40)

In [ ]:
df['condition'] = df['condition'] - 1

In [ ]:
df

In [ ]:
# Afficher le type de chaque colonne
print(df.dtypes)

In [ ]:
df.describe()

In [ ]:
import pandas as pd

# Transformer yr_built et renovated en datetime (1er janvier de l'année)
df['yr_built_date'] = pd.to_datetime(df['yr_built'], format='%Y')
df['renovated_date'] = pd.to_datetime(df['renovated'], format='%Y')

# Vérifier
print(df[['yr_built', 'yr_built_date', 'renovated', 'renovated_date']].head())

In [ ]:
df

In [ ]:
df = df.drop(columns = ["yr_built" ,"renovated"])
df

# 3.EDA & Descriptif Statistics 

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.scatter(
    df.index.get_level_values(0),
    df["price"],
    alpha=0.5
)
plt.xlabel("Date")
plt.ylabel("price")
plt.title("Scatter plot – Ventes dans le temps")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1️⃣ Transformer le temps en variable numérique
t = np.arange(len(df))   # 0, 1, 2, ..., T
y = df["price"].values

# 2️⃣ Estimation de la tendance (régression linéaire simple)
slope, intercept = np.polyfit(t, y, 1)

# 3️⃣ Valeurs de la tendance
trend = intercept + slope * t

# 4️⃣ Plot
plt.figure(figsize=(10,5))

plt.scatter(
    df.index.get_level_values(0),
    y,
    alpha=0.5,
    label="price"
)

plt.plot(
    df.index.get_level_values(0),
    trend,
    color="tab:red",
    linewidth=2,
    label=f"Tendance (pente = {slope:.3f})"
)

plt.xlabel("Date")
plt.ylabel("price")
plt.title("prix dans le temps avec tendance")
plt.legend()
plt.show()

In [ ]:
df_ts = df.groupby(level=0)["price"].mean()

plt.figure(figsize=(10,5))
plt.plot(df_ts)
plt.xlabel("Date")
plt.ylabel("prix moyennes")
plt.title("Évolution temporelle moyenne des ventes")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.scatter(
    df.index.get_level_values(0),
    df["sqft_living"],
    color="tab:orange",
    alpha=0.5
)
plt.xlabel("Date")
plt.ylabel("sqft_living")
plt.title("Scatter plot – sqft_living dans le temps")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1️⃣ Transformer le temps en variable numérique
t = np.arange(len(df))   # 0, 1, 2, ..., T
y = df["sqft_living"].values

# 2️⃣ Estimation de la tendance (régression linéaire simple)
slope, intercept = np.polyfit(t, y, 1)

# 3️⃣ Valeurs de la tendance
trend = intercept + slope * t

# 4️⃣ Plot
plt.figure(figsize=(10,5))

plt.scatter(
    df.index.get_level_values(0),
    y,
    color="tab:orange",
    alpha=0.5,
    label="sqft_living"
)

plt.plot(
    df.index.get_level_values(0),
    trend,
    color="tab:red",
    linewidth=2,
    label=f"Tendance (pente = {slope:.3f})"
)

plt.xlabel("Date")
plt.ylabel("sqft_living")
plt.title("sqft_living dans le temps avec tendance")
plt.legend()
plt.show()

In [ ]:
df_ts = df.groupby(level=0)["sqft_living"].mean()

plt.figure(figsize=(10,5))
plt.plot(df_ts, color="tab:orange")
plt.xlabel("Date")
plt.ylabel("sqft_living")
plt.title("Évolution temporelle moyenne des sqft_living")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.scatter(
    df.index.get_level_values(0),
    df["price"],
    alpha=0.5,
    label="Ventes"
)

plt.scatter(
    df.index.get_level_values(0),
    df["sqft_living"],
    alpha=0.5,
    label="sqft_living"
)

plt.xlabel("Date")
plt.ylabel("Valeur")
plt.title("price & sqft_living dans le temps")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Compter les occurrences par catégorie
counts = df['waterfront'].value_counts()

# Calcul des pourcentages
pourcentages = counts / counts.sum() * 100

# Graphe circulaire
plt.figure(figsize=(7,7))
pourcentages.plot(
    kind='pie',
    autopct='%1.1f%%',  # affiche les pourcentages
    startangle=90,
    labels=['Pas de waterfront', 'Avec waterfront'],  # optionnel
    colors=['skyblue', 'orange'],
    explode=[0, 0.1]  # pour faire ressortir la catégorie 1
)
plt.ylabel('')
plt.title("Répartition des maisons selon Waterfront (%)")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Compter les occurrences par catégorie et trier
view_counts = df['view'].value_counts().sort_index()
view_percent = view_counts / view_counts.sum() * 100

# Créer le graphique
plt.figure(figsize=(10,5))
bars = plt.bar(view_percent.index, view_percent.values, color='skyblue')

# Ajouter les pourcentages sur les barres (en noir)
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,  # centre horizontal
        height/2,                         # milieu vertical de la barre
        f'{height:.1f}%',                 # texte : pourcentage
        ha='center', va='center', color='black', fontsize=12
    )

plt.xlabel("Catégorie de view")
plt.ylabel("Pourcentage (%)")
plt.title("Répartition des maisons selon view (%)")
plt.xticks(rotation=0)
plt.show()

In [ ]:
df

In [ ]:
import pandas as pd

desc = df.describe().T   # transpose pour lisibilité

desc["skewness"] = df.skew(numeric_only=True)
desc["kurtosis"] = df.kurtosis(numeric_only=True)
desc["missing_%"] = df.isna().mean() * 100

desc

# 4.Analyse Bivarié des variables (Features and target)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Sélectionner uniquement les colonnes numériques (exclure les qualitatives)
quali_vars = ['waterfront', 'view', 'condition', 'grade', 'zipcode', 'isrenovat']
numeric_df = df.drop(columns=quali_vars)

# Calculer la matrice de corrélation
corr_matrix = numeric_df.corr()

In [ ]:
corr_matrix

In [ ]:
# Afficher la matrice avec seaborn
plt.figure(figsize=(12,10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matrice de corrélation (variables quantitatives)")
plt.show()

In [ ]:
from scipy.stats import ttest_ind

# Groupe 1 : pas de vue
group1 = df[df['waterfront'] == 0]['price']

# Groupe 2 : avec vue (1,2,3,4)
group2 = df[df['waterfront'] > 0]['price']

# Test t de Student
t_stat, p_value = ttest_ind(group1, group2, equal_var=False)

print("Statistique t :", t_stat)
print("p-value :", p_value)

In [ ]:
from scipy.stats import ttest_ind

# Groupe 1 : pas de vue
group1 = df[df['isrenovat'] == 0]['price']

# Groupe 2 : avec vue (1,2,3,4)
group2 = df[df['isrenovat'] > 0]['price']

# Test t de Student
t_stat, p_value = ttest_ind(group1, group2, equal_var=False)

print("Statistique t :", t_stat)
print("p-value :", p_value)

In [ ]:
from scipy.stats import f_oneway

for var in ['view', 'condition', 'grade']:
    print(f"\nVariable : {var}")
    
    groups = []
    
    for v in sorted(df[var].unique()):
        group = df[df[var] == v]['price']
        groups.append(group)
        
        print(f"  {var} = {v}")
        print(f"    Effectif : {len(group)}")
        print(f"    Moyenne price : {group.mean():.2f}")
        print(f"    Ecart-type : {group.std():.2f}")
    
    f_stat, p_value = f_oneway(*groups)
    
    print(f"\n  Statistique F : {f_stat:.4f}")
    print(f"  p-value : {p_value:.6f}")

# 5.Split Data 

In [ ]:
#df.drop(columns = ["zipcode"])

In [ ]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(1 , inplace=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split

# Séparer X et y
X = df.drop(columns=["price"])
y = df["price"]

In [ ]:
#"bedrooms" ,"sqft_basement" ,"sqft_living", "sqft_lot" ,"lot_Dif" ,"total_rooms" ,"bathrooms" ,"sqft_above", "yr_built_date"

In [ ]:
X

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, shuffle=False
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, shuffle=False
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

coords_train = scaler.fit_transform(X_train[["lat","long"]])
coords_valid = scaler.transform(X_val[["lat","long"]])
coords_test = scaler.transform(X_test[["lat","long"]])

In [ ]:
coords_train

In [ ]:
import joblib

# Sauvegarde du scaler
joblib.dump(scaler, "scaler_coords.pkl")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,6))

plt.scatter(
    X_train["long"],
    X_train["lat"],
    alpha=0.5
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Distribution géographique des observations")

plt.show()

In [ ]:
import warnings
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')

inertia = []
K_range = range(2,15)

for k in K_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(coords_train)
    inertia.append(kmeans.inertia_)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(K_range, inertia, marker='o')
plt.xlabel("Nombre de clusters (K)")
plt.ylabel("Inertia")
plt.title("Méthode du coude (Elbow Method)")
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

sil_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = kmeans.fit_predict(coords_train)
    
    score = silhouette_score(coords_train, labels)
    sil_scores.append(score)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(K_range, sil_scores, marker='o')
plt.xlabel("Nombre de clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Choix optimal de K (Silhouette)")
plt.show()

In [ ]:
kmeans_final = KMeans(
    n_clusters=8,
    n_init=10,
    random_state=42
)

X_train["geo_cluster"] = kmeans_final.fit_predict(coords_train)
X_val["geo_cluster"] = kmeans_final.predict(coords_valid)
X_test["geo_cluster"] = kmeans_final.predict(coords_test)

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(
    X_train["long"],
    X_train["lat"],
    c=X_train["geo_cluster"],
    cmap="viridis",
    alpha=0.6
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Clusters géographiques")

plt.show()

In [ ]:
import joblib

joblib.dump(kmeans_final, "kmeans_geo.pkl")

In [ ]:
X_train

In [ ]:
quali_vars = ['waterfront', 'view', 'condition', 'grade', 'isrenovat', 'zipcode', 'geo_cluster']
time_vars = ['yr_built_date', 'renovated_date']

num_vars = [col for col in X_train.columns 
            if col not in quali_vars]

In [ ]:
X_train['yr_built_date'] = X_train['yr_built_date'].dt.year
X_val['yr_built_date'] = X_val['yr_built_date'].dt.year
X_test['yr_built_date'] = X_test['yr_built_date'].dt.year

In [ ]:
X_train['renovated_date'] = X_train['renovated_date'].dt.year
X_val['renovated_date'] = X_val['renovated_date'].dt.year
X_test['renovated_date'] = X_test['renovated_date'].dt.year

In [ ]:
X_train[num_vars]

In [ ]:
X_train[num_vars].dtypes

In [ ]:
print("Nombre de +inf :", np.isinf(X_train[num_vars]).sum().sum())
print("Nombre de -inf :", np.isneginf(X_train[num_vars]).sum().sum())
print("Nombre de NaN  :", np.isnan(X_train[num_vars]).sum().sum())

In [ ]:
print("Nombre de +inf :", np.isinf(X_test[num_vars]).sum().sum())
print("Nombre de -inf :", np.isneginf(X_test[num_vars]).sum().sum())
print("Nombre de NaN  :", np.isnan(X_test[num_vars]).sum().sum())

In [ ]:
print("Nombre de +inf :", np.isinf(X_val[num_vars]).sum().sum())
print("Nombre de -inf :", np.isneginf(X_val[num_vars]).sum().sum())
print("Nombre de NaN  :", np.isnan(X_val[num_vars]).sum().sum())

In [ ]:
for col in num_vars:
    if np.isinf(X_train[col]).any():
        print("Inf dans :", col)

In [ ]:
print(X_train.columns)

In [ ]:
# ===============================
# 📌 Target Encoding - zipcode
# ===============================

# Calcul mean sur train
zip_mean = (
    pd.concat([X_train, y_train], axis=1)
    .groupby("zipcode")["price"]
    .mean()
)

# Mapping
X_train["zipcode_te"] = X_train["zipcode"].map(zip_mean)
X_val["zipcode_te"]   = X_val["zipcode"].map(zip_mean)
X_test["zipcode_te"]  = X_test["zipcode"].map(zip_mean)

# Fill unseen
global_mean = y_train.mean()

X_train["zipcode_te"] = X_train["zipcode_te"].fillna(global_mean)
X_val["zipcode_te"]   = X_val["zipcode_te"].fillna(global_mean)
X_test["zipcode_te"]  = X_test["zipcode_te"].fillna(global_mean)

In [ ]:
# Vérifier les premières lignes
X_train[["zipcode_te"]].head()

In [ ]:
# Appliquer log1p pour compresser les grandes valeurs
X_train["zipcode_te"] = np.log(X_train["zipcode_te"])
X_val["zipcode_te"]   = np.log(X_val["zipcode_te"])
X_test["zipcode_te"]  = np.log(X_test["zipcode_te"])

In [ ]:
# Vérifier les premières lignes
X_train[["zipcode_te"]].head()

In [ ]:
# Combien de valeurs uniques dans la colonne encodée
print("Nombre de valeurs uniques avant TE :", X_train["zipcode"].nunique())

In [ ]:
# Combien de valeurs uniques dans la colonne encodée
print("Nombre de valeurs uniques après TE :", X_train["zipcode_te"].nunique())

In [ ]:
# Drop original
X_train = X_train.drop(columns=["zipcode"])
X_val   = X_val.drop(columns=["zipcode"])
X_test  = X_test.drop(columns=["zipcode"])

In [ ]:
X_train

In [ ]:
quali_vars = ['waterfront', 'view', 'condition', 'grade', 'isrenovat', 'zipcode_te', 'geo_cluster']
time_vars = ['yr_built_date', 'renovated_date']

num_vars = [col for col in X_train.columns 
            if col not in quali_vars]

In [ ]:
X_train[num_vars].describe

In [ ]:
# ===============================
# 📌 1. Définition des groupes
# ===============================

feature_groups = {

    "Nombre_pieces": [
        "bedrooms",
        "bathrooms",
        "total_rooms"
    ],

    "Surface": [
        "sqft_living",
        "sqft_above",
        "sqft_basement",
        "sqft_living15",
        "M_sqft",
        "R_Prliv"
    ],

    "Lot": [
        "sqft_lot",
        "sqft_lot15",
        "lot_ratio",
        "lot_Dif",
        "lot_Sum"
    ],

    "Chronologie": [
        "yr_built_date",
        "house_age",
        "renovated_date",
        "isrenovat"
    ],

    "Qualite_Condition": [
        "grade",
        "condition"
    ],

    "Localisation": [
        "lat",
        "long",
        "zipcode",
        "waterfront",
        "view"
    ]
}

In [ ]:
from IPython.display import display, Markdown

for group, variables in feature_groups.items():
    display(Markdown(f"## 📂 {group}"))
    for var in variables:
        display(Markdown(f"- {var}"))

# 6.Feature Selection & Standarisation

In [ ]:
quali_vars = ['waterfront', 'view', 'condition', 'grade', 'isrenovat', 'zipcode_te', 'geo_cluster']
time_vars = ['yr_built_date', 'renovated_date']

num_vars = [col for col in X_train.columns 
            if col not in quali_vars]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[num_vars] = scaler.fit_transform(X_train[num_vars])

X_val[num_vars] = scaler.transform(X_val[num_vars])
X_test[num_vars] = scaler.transform(X_test[num_vars])

In [ ]:
means = scaler.mean_
stds = scaler.scale_

In [ ]:
import pandas as pd

scaler_params = pd.DataFrame({
    "Feature": num_vars,
    "Mean (μ)": means,
    "Std (σ)": stds
})

scaler_params

In [ ]:
X_train

In [ ]:
type(X_train)

In [ ]:
X_train = X_train.drop(columns = ['long', 'lat'])

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_Check = pd.DataFrame({
    "Features": X_train.columns,
    "VIF": [variance_inflation_factor(X_train, i) for i in range(X_train.shape[1])]
}).sort_values(by="VIF", ascending=False)
print(vif_Check)

In [ ]:
#Tant que ne depasse pas seuil de 5 en VIF ,on peut juger comme acceptable (15 features pour les modèles parametrique)

In [ ]:
X_train.dtypes

In [ ]:
X_train.shape

In [ ]:
X_train 

In [ ]:
X_train=X_train.values.reshape(15129, 26)

In [ ]:
type(X_train)

In [ ]:
X_train

In [ ]:
X_train.shape

In [ ]:
X_test = X_test.drop(columns = ['long', 'lat'])

In [ ]:
X_test

In [ ]:
X_val = X_val.drop(columns = ['long', 'lat'])

In [ ]:
X_val

In [ ]:
list(df.columns)

In [ ]:
type(X_test)

In [ ]:
type(X_val)

In [ ]:
X_test.dtypes

In [ ]:
X_val.dtypes

In [ ]:
X_test.shape

In [ ]:
X_val.shape

In [ ]:
X_test=X_test.values.reshape(3242, 26)

In [ ]:
X_val=X_val.values.reshape(3242, 26)

In [ ]:
type(X_test)

In [ ]:
type(X_val)

In [ ]:
X_test

In [ ]:
X_val

In [ ]:
X_test.shape

In [ ]:
X_val.shape

In [ ]:
type(y_train)

In [ ]:
type(y_test)

In [ ]:
type(y_val)

In [ ]:
print(type(X_train))
print(type(X_test))
print(type(X_val))

In [ ]:
print(X_train[:1, :30])   # première colonne

In [ ]:
print(y_train.shape)
print(y_test.shape)
print(y_val.shape)

In [ ]:
y_train=y_train.values.reshape(15129, 1)
y_test=y_test.values.reshape(3242, 1)
y_val=y_val.values.reshape(3242, 1)

In [ ]:
print(y_train.shape)
print(y_test.shape)
print(y_val.shape)

In [ ]:
print(type(y_train))
print(type(y_test))
print(type(y_val))

# 7.Regression Multiple : OLS, Decision Tree, Random forest, XGBoost ,LigthGBM

In [ ]:
%pip install xgboost
%pip install lightgbm

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, root_mean_squared_error

In [ ]:
#9.1 Regression Multiple : OLS, Decision Tree, Random forest

In [ ]:
model = LinearRegression()

model.fit(X_train, y_train)

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [ ]:
print("R² Train =", r2_score(y_train, y_pred_train))
print("R² Test =", r2_score(y_test, y_pred_test))

print("RMSE Train =", root_mean_squared_error(y_train, y_pred_train))
print("RMSE Test =", root_mean_squared_error(y_test, y_pred_test))

In [ ]:
model = DecisionTreeRegressor(
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [ ]:
print("R² Train =", r2_score(y_train, y_pred_train))
print("R² Test =", r2_score(y_test, y_pred_test))

print("RMSE Train =", root_mean_squared_error(y_train, y_pred_train))
print("RMSE Test =", root_mean_squared_error(y_test, y_pred_test))

In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [ ]:
print("R² Train =", r2_score(y_train, y_pred_train))
print("R² Test =", r2_score(y_test, y_pred_test))

print("RMSE Train =", root_mean_squared_error(y_train, y_pred_train))
print("RMSE Test =", root_mean_squared_error(y_test, y_pred_test))

In [ ]:
model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [ ]:
print("R² Train =", r2_score(y_train, y_pred_train))
print("R² Test =", r2_score(y_test, y_pred_test))

print("RMSE Train =", root_mean_squared_error(y_train, y_pred_train))
print("RMSE Test =", root_mean_squared_error(y_test, y_pred_test))

In [ ]:
model = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=-1,
    num_leaves=31,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [ ]:
print("R² Train =", r2_score(y_train, y_pred_train))
print("R² Test =", r2_score(y_test, y_pred_test))

print("RMSE Train =", root_mean_squared_error(y_train, y_pred_train))
print("RMSE Test =", root_mean_squared_error(y_test, y_pred_test))

#  7.Validation Croisee de XGBoost & Deployement

In [ ]:
xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=1.2,
    random_state=42
)

xgb.fit(X_train, y_train)

In [ ]:
y_pred_train = xgb.predict(X_train)
y_pred_test = xgb.predict(X_test)

print("R² Train =", r2_score(y_train, y_pred_train))
print("R² Test =", r2_score(y_test, y_pred_test))

print("RMSE Train =", root_mean_squared_error(y_train, y_pred_train))
print("RMSE Test =", root_mean_squared_error(y_test, y_pred_test))

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

r2_train = []

for n in range(1, 101):

    model = XGBRegressor(
        n_estimators=n,
        learning_rate=1.2,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_train)

    r2_train.append(
        r2_score(y_train, y_pred)
    )

plt.figure(figsize=(8,5))

plt.plot(
    range(1,101),
    r2_train
)

plt.xlabel("Nombre d'itérations (n_estimators)")
plt.ylabel("R² Train")
plt.title("Courbe d'apprentissage XGBoost")

plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error

y_pred_val = xgb.predict(X_val)

print("R² Validation =", r2_score(y_val, y_pred_val))
print("RMSE Validation =", root_mean_squared_error(y_val, y_pred_val))

In [ ]:
y_train_pred = xgb.predict(X_train)

print("R² Train =", r2_score(y_train, y_pred_train))
print("R² Test =", r2_score(y_test, y_pred_test))
print("R² Validation =", r2_score(y_val, y_pred_val))

In [ ]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

cv = KFold(
    n_splits=5,
    shuffle=False
)

scores = cross_val_score(
    xgb,
    X_train,
    y_train,
    cv=cv,
    scoring='r2'
)

print(scores)
print(scores.mean())
print(scores.std())